# ShrRep3 — 3-Party Replicated Secret Sharing (ABY3)

ELL ∈ [1, 6].  Three-party protocol over a ring topology
(P0 → P1 → P2 → P0).  Each party holds **two** out of three
replicated shares.

## Factory

``ShrRep3(ell, party)`` returns a **type** (not an instance).
Call it with ``(ch_prev, ch_next)`` to construct a protocol instance.
``ch_prev`` receives from the previous party; ``ch_next`` sends to
the next party.


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

import mpmt
from mpmt.channels import _build_rep3_channels

ch = _build_rep3_channels(
    prev_port=14000, next_host="127.0.0.1", next_port=14001,
    party_id=0,
)

Rep3 = mpmt.ShrRep3(ell=4, party=0)
inst = Rep3(ch["prev"], ch["next"])
print(f"ell={inst.ell}, party={inst.party}")


## Ring Setup

``_build_rep3_channels(prev_port, next_host, next_port, party_id)``
establishes the ring without daemon threads.  P0 and P2 accept first
then connect; P1 connects first then accepts — breaking the circular
dependency.  Returns ``{"prev": Channel, "next": Channel}``.


## Scalar Sharing

**Leader (P0)**: ``inst.share_scalar(val)`` — generates RSS3 shares
of *val*, sends them through the ring.

**Helpers (P1, P2)**: ``inst.recv_scalar_share()`` — receives the
share from the ring.

Returns a ``ShrRep3ShareScalar`` with ``this_share`` and ``nxt_share``
fields (each a ``uint8_t``).


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

# All three parties call these in order:
if pid == 0:
    ss = inst.share_scalar(val=7)
else:
    ss = inst.recv_scalar_share()
# ss.this_share, ss.nxt_share


## Vector Sharing

**Leader**: ``inst.share_vector(vec, sv, auxBuf)`` — shares the
``Rvector`` *vec*; writes the result into *sv* (pre-allocated
``ShrRep3ShareVec``).  *auxBuf* is an ``RvectorPack`` scratch buffer.

**Helpers**: ``inst.recv_vector_share(sv, auxBuf)`` — receives
the share vector.


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

SV = mpmt.ShrRep3ShareVec(ell=4)
aux = mpmt.RvectorPack(ell=4)(bf_size)

if pid == 0:
    vec = mpmt.Rvector(ell=4)(bf_size)
    vec.fill(val=1)
    sv = SV(bf_size)
    inst.share_vector(vec, sv, auxBuf=aux)
else:
    sv = SV(bf_size)
    inst.recv_vector_share(sv, auxBuf=aux)


## Reshare

**share** takes a *plaintext* value from P0 and converts it to RSS3.
**reshare** takes an *existing additive share* (already held by every
party — e.g. DPF output + crng) and redistributes it through the ring
to re-establish 2-of-3 replication.  No party holds the plaintext;
every party already has its own additive component.

Scalar and vector variants are available and symmetric across all
three parties — there is no leader/helper distinction.

> **Aliasing**: ``reshare_vector(vec, sv, auxBuf)`` — *vec* may alias
> ``sv.nxt_share``, but **not** ``sv.this_share`` (the latter is the
> send buffer and must not be overwritten before the ring round).

In [ ]:
# This cell requires multi-party setup and will not run in a notebook

# ——— Scalar ———
# All three parties call reshare_scalar with their additive byte:
additive = my_additive_byte(some_input)
ss = inst.reshare_scalar(val=additive)
# ss.this_share, ss.nxt_share  — now in RSS3 form

# ——— Vector ———
# additive_vec is each party's own additive Rvector component
sv = SV(bf_size)
inst.reshare_vector(vec=additive_vec, sv=sv, auxBuf=aux)
# sv.this_share, sv.nxt_share  — now in RSS3 form

## Reveal

``inst.reveal_scalar(ss)`` — reconstruct a scalar share.
All three parties learn the result.

``inst.reveal_vector(sv, out, auxBuf)`` — reconstruct a vector share.
*out* must be a pre-allocated ``Rvector`` of matching size.


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

out = mpmt.Rvector(ell=4)(bf_size)
inst.reveal_vector(sv, out, auxBuf=aux)
# out[i] is now the plaintext


## Arithmetic

- ``inst.add_vec(sv1, sv2, out)`` — local (no network)
- ``inst.sub_vec(sv1, sv2, out)`` — local
- ``inst.hadamard(sv1, sv2, out)`` — element-wise multiply (1 round)
- ``inst.dot(sv1, sv2)`` — inner product → ShareScalar (1 round)
- ``inst.add(ss1, ss2)``, ``inst.sub(ss1, ss2)``, ``inst.mul(ss1, ss2)`` — scalar variants


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

# Vector: out must not alias inputs for hadamard
inst.add_vec(sv1, sv2, out)
inst.hadamard(sv1, sv2, tmp)
inst.sub_vec(out, tmp, out)

# Scalar
ss3 = inst.add(ss1, ss2)


## Correlated Randomness (crng)

``inst.crng()`` — single random byte where all three parties' values
sum to zero.

``inst.crng_vec(vec)`` — vector variant.  Used in the GenBF protocol
to convert DPF output (2-of-2 share) to a 3-of-3 additive share:
each party adds its crng vector to its DPF share component, producing
three vectors that sum to the plaintext Bloom filter.

$$r_{P_0} + r_{P_1} + r_{P_2} \equiv 0 \pmod{2^{\,ELL}}$$

In [ ]:
# This cell requires multi-party setup and will not run in a notebook

# All three parties call simultaneously:
r = inst.crng()              # single byte:  r_P0 + r_P1 + r_P2 = 0
rv = mpmt.Rvector(ell=4)(bf_size)
inst.crng_vec(rv)            # vector: each element sums to 0 across parties

## Ring Conversion

``inst.ring_conv(ss, ell_to)`` — scalar: convert a binary (ELL=1)
share to arithmetic (ELL 2–6).

``inst.ring_conv_vec(sv_ell1, sv_out, ell_to)`` — vector variant.
*svo_out* must be a pre-allocated ``ShrRep3ShareVec`` of ELL=*ell_to*.
Available only on ELL=1 instances.

Used before the dot product: the root BF (binary, ELL=1) must be
converted to the arithmetic ring (ELL=ell_query) to compute the
inner product with the query BF.


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

# Only on ELL=1 instance:
sv_q = mpmt.ShrRep3ShareVec(ell=4)(bf_size)
inst_ell1.ring_conv_vec(root_sv, sv_q, ell_to=4)


## Byte Counters

- ``inst.bytes_sent()``, ``inst.bytes_recv()``
- ``inst.clear_send_cnt()``, ``inst.clear_recv_cnt()``
- ``inst.flush()`` — drain pending sends
